# Figure 2 — from along-scan detections to a sky position

**In the paper:** Fig. 2; Sect. 3 (Eqs. 9–13); Sect. 4.3 (Eq. 18); the numbers in Sect. 4.4.

A static companion, measured from the 47 windows of one campaign: each window gives a position
along its scan only, and the scans together fix the position on the sky.

| step | task | calls | paper |
|---|---|---|---|
| 1 | the scan law and the scene; the projection onto a scan | `scanlaw.load_scan_law`, `sim.Scene.pair` | Sect. 3.1, Eq. 9 |
| 2 | the windows | `sim.simulate_window` | Sect. 2.1, Eq. 4; Sect. 4.4 |
| 3 | the per-window reduction: the detections, and the censored windows | `red.ladder_window`, `red.delivery_mask`, `red.sigma_eta`; in one call `sim.harvest_campaign` | Sect. 2.2, Fig. 1 |
| 4 | the sinogram and its fit, panel (b) | `joint.polar_elements`, `ins.project_al` | Sect. 3.2, Eqs. 11, 12 |
| 5 | the triangulation map, panel (c), and the seed of the fit: the map's peak selects the detections, the sinogram fit refines | `bp.backprojection_map`, `bp.map_peak`, `bp.peak_consistency`, `dyn.motion_design`, `dyn.wls`; in one call `dyn.clean_detection_set` | Sect. 3.3, Eq. 13 |
| 6 | the window-level likelihood | `joint.scene_positions_al`, `joint.loglike_profiled`, `joint.loglike_windows`, `joint.stacked_design` | Sect. 4.3, Eq. 18 |
| 7 | the fit | `opt.minimize_restarts`, `joint.joint_fisher`, `joint.joint_covariance`; in one call `joint.fit_joint` | Sect. 4.3 |
| 8 | the uncertainties, and the inset of panel (c) | `joint.loglike_profiled` | Sect. 4.4 |
| 9 | the two transits the figure highlights | `ins.project_al` | — |
| 10 | the figure | — | Fig. 2 |

Every scanlib call carries the name of its module (`dyn.wls`, `joint.fit_joint`); the first code cell lists the
modules.

**Paper and code.** Triangulation map = `bp.backprojection_map`; flux ratio 𝒮 = `Q`; the
sinogram crossing = `seed`; Λ_w = `margins`. Where the package has a one-call form of a stage
(`sim.harvest_campaign`, `dyn.clean_detection_set`, `joint.fit_joint`), the notebook asserts that it agrees
with the stage written out.

**What it reproduces** (Sect. 4.4). 29 of the 47 windows give a companion detection, 18 are
fitted with one source; all 29 are consistent with the map candidate; the fit gives
(189.8 ± 1.3, −69.0 ± 0.9) mas against the injected (187.9, −68.4), within about 2 mas, and
𝒮 = 0.197 ± 0.003. A few seconds; `SAVE_PDF = True` writes the PDF.

In [ ]:
# the imports: numpy, matplotlib, and the scanlib modules; every scanlib call below carries its module's name
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from scanlib import instrument as ins      # the instrument: sampling, LSF, the along-scan projection (Eq. 9)
from scanlib import scanlaw                # scan angles, epochs and parallax factors
from scanlib import simulation as sim      # synthetic scenes and their windows
from scanlib import reduction as red       # one window, or a campaign, reduced to detections
from scanlib import backprojection as bp   # the triangulation map and its peak
from scanlib import dynamics as dyn        # the fits to the detections (the sinogram fits)
from scanlib import joint                  # the joint fit of all windows, and its helpers
from scanlib import optimize as opt        # the optimizer of the joint fit

In [ ]:
# the instrument, defined once (Sect. 2.1): 18 samples of 59 mas along scan, a Gaussian LSF of FWHM 100 mas,
# and its per-window margins Delta_1, Delta_2 (App. A.2; derived in auxil/sim_margins_lambda_w)
FWHM_MAS = 100.0                           # the full width at half maximum of the LSF [mas]
PITCH_MAS = 59.0                           # the width of one sample along the scan [mas]
N_SAMPLES = 18                             # the number of samples of one window
inst = ins.Instrument.gaussian(fwhm_mas=FWHM_MAS, pitch_mas=PITCH_MAS, n_samples=N_SAMPLES, margins=ins.TOY_MARGINS)

SAVE_PDF = False                           # True: write pdf/fig2_scans_to_sky.pdf

## 1. The scan law and the scene (Sect. 3.1, Eq. 9; Sect. 4.4)

A transit scans the field at angle ψ (from north through east). A source at sky offset
(Δα\*, Δδ) = ρ (sin φ, cos φ) appears along the scan at

$$\eta(\psi) = \Delta\alpha^{*}\sin\psi + \Delta\delta\cos\psi = \rho\cos(\psi-\varphi). \qquad (9)$$

The primary is at the origin of every window; the companion does not move, so the epochs are
not used. The scene is a pair, so each window is reduced to two sources at most.

In [ ]:
# the scene: a static companion 200 mas from the primary; the scan law; and the along-scan position of both
# sources in each of the 47 transits (Eq. 9)
RHO_MAS = 200.0                            # the companion's separation [mas]
PA_DEG = 110.0                             # its position angle [deg], from north through east
Q = 0.2                                    # its flux ratio, companion over primary
SEED = 2                                   # this figure's noise realization
K_MAX = 2                                  # at most two sources per window

# sim.Scene.pair: the primary at the origin, the companion at (rho, PA), flux ratio Q
scene = sim.Scene.pair(RHO_MAS, PA_DEG, Q)
# the Pleiades-centre law, named explicitly: psi, the scan angle of each transit [rad], and t_years, its epoch
# [yr from the reference epoch]. The companion is static, so t_years is not used below
psi, t_years = scanlaw.load_scan_law(scanlaw.PLEIADES)
# positions has one row (dra*, ddec) per source; row 1 is the companion (row 0 is the primary)
companion_ra, companion_dec = scene.positions[1]     # the companion's truth [mas]
# along_scan: Eq. 9 for each source in one transit. eta_true has one row per transit and two columns, shape
# (47, 2): column 0 the primary, column 1 the companion
eta_per_transit = []
for psi_j in psi:
    eta_per_transit.append(scene.along_scan(psi_j))
eta_true = np.array(eta_per_transit)

In [ ]:
# the scene checked and printed: 47 transits, and Eq. 9 by hand against the scene's own projection
assert psi.size == 47
eta_true_primary = eta_true[:, 0]          # every transit, column 0: the primary [mas]
eta_true_companion = eta_true[:, 1]        # every transit, column 1: the companion [mas]
# Eq. 9 by hand: the companion's eta is rho cos(psi - phi) in every transit
assert np.allclose(eta_true_companion, RHO_MAS * np.cos(psi - np.deg2rad(PA_DEG)), rtol=0, atol=1e-9)
assert np.all(eta_true_primary == 0.0)                   # the primary sits at the window origin
print(f"transits                       {psi.size}")
# the scan angles in degrees, folded to 0-180 (a scan and its reverse are one direction), sorted, as integers
print(f"scan angles (mod 180) [deg]    {np.sort(np.rad2deg(psi) % 180).round(0).astype(int)}")
print(f"companion offset [mas]         ({companion_ra:.1f}, {companion_dec:.1f})   "
      f"rho {RHO_MAS:.0f}, PA {PA_DEG:.0f} deg")
print(f"fluxes (primary, companion)    {scene.fluxes}   q = {Q}")
print(f"companion |eta| [mas]          min {np.abs(eta_true_companion).min():.1f}, "
      f"max {np.abs(eta_true_companion).max():.1f}")

## 2. The windows (Sect. 2.1, Eq. 4; Sect. 4.4)

Window $j$ is the scene through the LSF, sampled on 18 pixels, plus white noise:
$\vec s_j = \mathbf A_j(\boldsymbol\eta_j)\,\vec f + \vec\varepsilon_j$. The noise per sample is
σ_n = 10, set by S/N 100 for the isolated primary, the same in every window.

In [ ]:
# the 47 noisy windows, one draw each from one generator, in the order of the scan law
SNR_PRIMARY = 100.0                        # the signal-to-noise ratio of the isolated primary (total flux over sigma)
sigma_noise = sim.PRIMARY_FLUX / SNR_PRIMARY   # the noise sigma per sample [flux per sample]
# the generator of the noise. Its seed is one number, this figure's realization; nothing else enters it
rng_noise = np.random.default_rng(SEED)
windows = []
for eta_j in eta_true:                         # eta_j: the two sources' along-scan positions in transit j [mas]
    # sim.simulate_window: the scene through the LSF on 18 samples, plus noise
    samples = sim.simulate_window(eta_j, scene.fluxes, inst, rng_noise, sigma_noise)
    windows.append(samples)

assert sigma_noise == 10.0
print(f"windows                        {len(windows)} x {windows[0].size} samples")
print(f"noise sigma per sample         {sigma_noise:.3f}")

## 3. The per-window reduction (Sect. 2.2; Fig. 1)

Each window alone, as in Fig. 1: one source, then two if Δχ² > Λ_w. Of the fitted sources,
the one nearest the window origin is the primary; the other is a **detection**
(ψ, η̂, σ_η). A window reduced to one source gives no detection: it is **censored**, and it
still enters the fit of step 7.

In [ ]:
# reduce each window alone (the ladder of Fig. 1) and keep, as detections (psi, eta-hat, sigma_eta), every fitted
# source but the primary; a window reduced to one source gives none and is censored
# inst.margins holds Delta_1 and Delta_2; [:K_MAX - 1] keeps the first K_MAX - 1 of them, here Delta_1 alone
margins = inst.margins[:K_MAX - 1]         # Delta_1: one rung, at most K_MAX sources

n_sources_list = []                        # one entry per window: the number of sources the ladder selected
window_list = []                           # one entry per detection: the number of its window
psi_list = []                              # one entry per detection: its window's scan angle [rad]
eta_list = []                              # one entry per detection: the detected position [mas]
sigma_list = []                            # one entry per detection: the error of that position [mas]
for j, samples in enumerate(windows):      # j counts the windows
    # red.ladder_window: one source, then two if the fit improves by more than Delta_1. It returns the positions,
    # the fluxes, the number of sources, and a dictionary of diagnostics (dropped here, the name _)
    eta, flux, k, _ = red.ladder_window(samples, inst, sigma_noise, k_max=K_MAX)
    n_sources_list.append(k)
    # red.delivery_mask: True for every fitted source but the one nearest the origin (the primary)
    delivered = red.delivery_mask(eta)
    if delivered.any():
        # red.sigma_eta: each source's along-scan error, from the window's Fisher matrix
        sigma_eta = red.sigma_eta(eta, flux, inst, sigma_noise)
        for eta_one, sigma_one in zip(eta[delivered], sigma_eta[delivered]):
            window_list.append(j)
            psi_list.append(psi[j])
            eta_list.append(eta_one)
            sigma_list.append(sigma_one)

In [ ]:
# the detections as arrays: their window, psi, eta-hat and sigma_eta; a sigma that is not finite gets the
# package's fallback
n_sources = np.array(n_sources_list)       # per window: the number of sources the ladder selected

det_window = np.array(window_list)         # per detection: the number of its window
det_psi = np.array(psi_list)               # per detection: the scan angle [rad]
det_eta = np.array(eta_list)               # per detection: the detected along-scan position [mas]
det_sigma = np.array(sigma_list)           # per detection: the error of that position [mas]
n_substituted = int((~np.isfinite(det_sigma)).sum())   # the number of errors that are not finite
# a singular Fisher matrix gives an error that is not finite: it gets the package's fallback value
det_sigma = np.where(np.isfinite(det_sigma), det_sigma, red.SIGMA_DET_FALLBACK_MAS)

In [ ]:
# the censored windows, the numbers of Sect. 4.4 asserted, and a summary of the reduction
# np.setdiff1d: the window numbers 0 to 46 that are not the window of any detection
censored = np.setdiff1d(np.arange(psi.size), det_window)
# np.unique keeps each window's number once: the windows that gave a detection
n_windows_detected = np.unique(det_window).size
assert det_eta.size == 29 and n_windows_detected == 29 and censored.size == 18   # Sect. 4.4: 29 and 18
assert np.all(n_sources[censored] == 1)
print(f"windows yielding a companion   {n_windows_detected} of {len(windows)}")
print(f"windows censored               {censored.size}   (all reduced to one source)")
print(f"ladder margin Delta_1          {margins[0]:.2f}")
line = f"sigma_eta median / max [mas]   {np.median(det_sigma):.1f} / {det_sigma.max():.1f}"
if n_substituted:                          # the note is printed only when an error was substituted
    line = line + f"   ({n_substituted} substituted)"
print(line)

In [ ]:
# the convenience: sim.harvest_campaign renders and reduces the whole campaign in one call, from the same seed; it must
# give the same windows and detections as the steps above
# the generator has the seed of section 2, so sim.harvest_campaign draws the same noise
rng_harvest = np.random.default_rng(SEED)
harvest = sim.harvest_campaign(scene, psi, rng_harvest, inst, sigma_noise, k_max=K_MAX, return_windows=True)
# the windows are equal one by one (a pairs with b)
assert all(np.array_equal(a, b) for a, b in zip(harvest.windows, windows))
assert harvest.sigma_noise == sigma_noise
# each array of the harvest (a) against the same array written out above (b)
for a, b in ((harvest.psi, det_psi), (harvest.eta, det_eta), (harvest.sigma, det_sigma)):
    assert np.array_equal(a, b)
print("sim.harvest_campaign reproduces the windows and the detections exactly")

### The censored windows

The scans that run nearly across the pair: there the projected separation is a small fraction of
the FWHM.

In [ ]:
# the censored windows, by scan angle, with the companion's true projected separation on each scan
# np.argsort gives the order that sorts the censored windows by their scan angle [deg, 0 to 360]
censored_by_angle = censored[np.argsort(np.rad2deg(psi[censored]) % 360)]
print(" psi [deg]   true |eta| [mas]")
for j in censored_by_angle:                # j: the window's number; eta_true[j, 1] is the companion's true eta there
    print(f"{np.rad2deg(psi[j]) % 360:9.1f} {abs(eta_true[j, 1]):14.1f}")
print(f"largest true separation of a censored window: "
      f"{np.abs(eta_true[censored, 1]).max() / FWHM_MAS:.2f} FWHM")

## 4. The sinogram and its fit (Sect. 3.2, Eqs. 11, 12), panel (b)

The 29 detections against the scan angle trace the sinusoid of Eq. 9. A static companion is a
linear model in its two offsets,

$$\hat\eta_i = a_1\sin\psi_i + a_2\cos\psi_i + \varepsilon_i,\quad \varepsilon_i\sim\mathcal N(0,\sigma_{\eta,i}^2), \qquad (11)$$

solved by weighted least squares, $\hat{\vec a} = (\mathbf X^{\mathsf T}\mathbf W\mathbf X)^{-1}\mathbf X^{\mathsf T}\mathbf W\hat{\vec\eta}$
with rows $(\sin\psi_i, \cos\psi_i)$ and $\mathbf W = {\rm diag}(\sigma_{\eta,i}^{-2})$; then
$\hat\rho = \sqrt{a_1^2+a_2^2}$, $\hat\varphi = {\rm atan2}(a_1, a_2)$ (Eq. 12). Detections from blended
scans are pulled toward the primary; this fit only starts the joint fit of step 7.

In [ ]:
# Eq. 11 written out: the design matrix, the weights, the normal equations
X = np.column_stack([np.sin(det_psi), np.cos(det_psi)])                # rows (sin psi_i, cos psi_i)
W = np.diag(1.0 / det_sigma ** 2)                                      # the diagonal matrix of the weights 1/sigma^2
a_hat = np.linalg.solve(X.T @ W @ X, X.T @ W @ det_eta)                # (a_1, a_2) of Eq. 11: the fitted offsets [mas]
a_cov = np.linalg.inv(X.T @ W @ X)                                     # their covariance [mas^2]
print("X, first three rows:\n", X[:3].round(3))
# a_cov[0, 0] and a_cov[1, 1] are the variances of a_1 and a_2; their roots are the 1-sigma errors
print(f"a_hat = ({a_hat[0]:.1f} +/- {np.sqrt(a_cov[0, 0]):.1f}, {a_hat[1]:.1f} +/- {np.sqrt(a_cov[1, 1]):.1f}) mas")

In [ ]:
# Eq. 12: the fit as a separation and a position angle, with their errors
# joint.polar_elements: Eq. 12, errors by linear propagation; rho [mas] and the position angle [deg]
rho_hat, sig_rho, phi_hat, sig_phi = joint.polar_elements(a_hat, a_cov)
print(f"rho = {rho_hat:.1f} +/- {sig_rho:.1f} mas, PA = {phi_hat:.1f} +/- {sig_phi:.1f} deg   "
      f"(truth {RHO_MAS:.0f}, {PA_DEG:.0f})")
# Eq. 12 by hand: np.hypot(*a_hat) is sqrt(a_1^2 + a_2^2); np.arctan2(*a_hat) is atan2(a_1, a_2)
assert np.isclose(rho_hat, np.hypot(*a_hat)) and np.isclose(phi_hat, np.rad2deg(np.arctan2(*a_hat)) % 360.0)

In [ ]:
# truth: the detections' residuals from the true track, and their pulls
# ins.project_al: Eq. 9; the true track is the companion's true eta at each detection's scan angle
eta_true_track = ins.project_al(companion_ra, companion_dec, det_psi)
residual = det_eta - eta_true_track                                     # residuals from the TRUE track [mas]
pull = residual / det_sigma                                             # the residual in units of its error
print(f"residual from the true track   median |r| {np.median(np.abs(residual)):.1f} mas, "
      f"max {np.abs(residual).max():.1f} mas")
print(f"pulls r / sigma_eta            rms {np.sqrt(np.mean(pull**2)):.2f}, max |pull| {np.abs(pull).max():.1f}")

## 5. The triangulation map (Sect. 3.3, Eq. 13), panel (c), and the seed

Each detection allows the sky line $\Delta\alpha^*\sin\psi_i + \Delta\delta\cos\psi_i = \hat\eta_i$ (Eq. 10).
The map adds, at each sky position $\vec x$, a unit-area Gaussian in its distance $\delta_{\vec x,i}$
from every line:

$$M_{\vec x} = \sum_i \frac{1}{\sqrt{2\pi}\,\sigma_{\eta,i}}\exp\!\left(-\frac{\delta_{\vec x,i}^2}{2\sigma_{\eta,i}^2}\right). \qquad (13)$$

**The seed, in three steps.** (1) *The map detects:* take the map's strongest peak outside 30 mas of the
primary. (2) *Select:* keep the detections whose line passes within max(3σ_η, the peak's width) of the peak;
the others do not belong to this source. (3) *The sinogram refines:* fit Eq. 11 to the kept detections alone.
That fit is the **seed** of step 7. Here every one of the 29 lines passes through the peak, so the seed equals
the fit of step 4.

For a **moving** companion (Fig. 3) the same three steps repeat: the fit of (3) also gives the motion, the
detections are moved to the reference epoch with it (Eq. 17), and the map of (1) is built again from them, until
the kept set stops changing. `dyn.clean_detection_set` is that loop; with `static=True` nothing moves the
detections, and it reduces to the three steps here.

In [ ]:
# Eq. 13 by hand at one pixel of the map (the pixel nearest the true companion) against the package's map
MAP_HALF_SIZE_MAS = 350.0                  # half the side of the map [mas]
MAP_STEP_MAS = 2.0                         # the pixel size of the map [mas]
# bp.backprojection_map: M of Eq. 13 on a grid of 2 mas pixels, its axis [mas] (the same for alpha* and delta), and
# its extent (dropped here, the name _)
sky_map_check, axis_check, _ = bp.backprojection_map(det_psi, det_eta, det_sigma, MAP_HALF_SIZE_MAS, MAP_STEP_MAS)
# the pixel nearest the true companion: its row (delta) and its column (alpha*)
i_dec = int(np.argmin(np.abs(axis_check - companion_dec)))
i_ra = int(np.argmin(np.abs(axis_check - companion_ra)))
# the distance of that pixel from each detection's line [mas]
distance = axis_check[i_ra] * np.sin(det_psi) + axis_check[i_dec] * np.cos(det_psi) - det_eta
M_hand = np.sum(np.exp(-0.5 * (distance / det_sigma) ** 2) / (np.sqrt(2.0 * np.pi) * det_sigma))
print(f"M at ({axis_check[i_ra]:.0f}, {axis_check[i_dec]:.0f}) mas: by hand {M_hand:.6f}, "
      f"bp.backprojection_map {sky_map_check[i_dec, i_ra]:.6f}")
assert np.isclose(M_hand, sky_map_check[i_dec, i_ra], rtol=1e-12)

In [ ]:
# the seed, steps (1)-(3) of the text: the map detects, its peak selects the detections, the sinogram fit refines
ORIGIN_ZONE_MAS = 30.0                     # step (1): no peak is taken within this distance of the primary [mas]
K_SIGMA = 3.0                              # step (2): a line may miss the peak by this many of its error bars
# dyn.motion_design returns four columns; [:, :2] keeps every row and the first two columns, [sin psi, cos psi]
# of Eq. 11, so we omit the two proper-motion columns (the companion is static)
epochs_not_used = np.zeros_like(det_psi)   # zeros in place of the epochs: the two columns they make are omitted
design = dyn.motion_design(det_psi, epochs_not_used)[:, :2]
weight = 1.0 / det_sigma ** 2              # the weight of each detection, 1/sigma^2

# (1) bp.map_peak: the map's highest peak outside 30 mas of the primary, and its width; it also returns the map
# and its axis (dropped here, the two _)
peak, width, _, _ = bp.map_peak(det_psi, det_eta, det_sigma, origin_zone=ORIGIN_ZONE_MAS)
# (2) bp.peak_consistency: True for each detection whose line passes within max(3 sigma, width) of the peak; it
# also returns each line's distance from the peak (dropped here, the name _)
keep, _ = bp.peak_consistency(det_psi, det_eta, det_sigma, peak, width, k_sigma=K_SIGMA)
# (3) dyn.wls: the weighted least-squares fit of Eq. 11 to the kept detections, and its covariance
seed, seed_cov = dyn.wls(design[keep], det_eta[keep], weight[keep])
print(f"peak ({peak[0]:.0f}, {peak[1]:.0f}) mas, width {width:.1f} mas, kept {keep.sum()} of {keep.size}")

In [ ]:
# Sect. 4.4: every detection is consistent with the map's candidate; with none rejected, the seed is the fit
# of step 4
assert keep.all()                          # Sect. 4.4: all 29 detections consistent with the map candidate
assert np.allclose(seed, a_hat, rtol=1e-12)  # with none rejected, the seed is the fit of step 4
# seed_cov[0, 0] and seed_cov[1, 1] are the variances of the two offsets; their roots are the 1-sigma errors
print(f"seed (the sinogram crossing)   ({seed[0]:.1f}, {seed[1]:.1f}) "
      f"+/- ({np.sqrt(seed_cov[0, 0]):.1f}, {np.sqrt(seed_cov[1, 1]):.1f}) mas;  "
      f"truth ({companion_ra:.1f}, {companion_dec:.1f})")

In [ ]:
# the convenience: dyn.clean_detection_set does the three steps in one call (static=True: no motion, no Eq. 17)
# it returns the seed, its covariance (dropped), the kept set, and a dictionary on how it ran (dropped); None
# stands for the epochs, which a static fit does not need
seed_one_call, _, keep_one_call, _ = dyn.clean_detection_set(det_psi, None, det_eta, det_sigma, static=True)
assert np.array_equal(seed_one_call, seed) and np.array_equal(keep_one_call, keep)
print("dyn.clean_detection_set reproduces the seed and the kept set exactly")

## 6. The window-level likelihood (Sect. 4.3, Eq. 18)

The measurement goes back to the samples of every window, the censored ones included. For the
companion at θ = (Δα\*, Δδ) and the primary at the origin, the model of window $j$ is
$\mathbf A_j(\theta)\vec f$, and

$$\ln\mathcal L(\theta, \vec f) = -\frac{1}{2\sigma_n^2}\sum_j \lVert \vec s_j - \mathbf A_j(\theta)\vec f\rVert^2 + {\rm const}. \qquad (18)$$

The fluxes enter linearly: at each θ their best non-negative values follow from one linear
solve (`joint.loglike_profiled` with `return_fluxes=True`), which leaves the **profiled** likelihood, a function of θ alone.
`joint.loglike_windows` is the same likelihood at fluxes given by hand, and `joint.stacked_design` is the matrix
$\mathbf A(\theta)$ of all windows stacked, one row per sample; the cell checks that the three forms agree.

In [ ]:
# the profiled likelihood of Eq. 18 at the truth and at the seed
truth = np.array([companion_ra, companion_dec])                     # theta at the truth: (dra*, ddec) [mas]
print("eta of each source, first three windows [mas]:")             # Eq. 9 for both sources at a trial theta
print(joint.scene_positions_al(truth, psi[:3]).round(1))            # psi[:3] keeps the first three windows

# ln L with the best fluxes f >= 0; with return_fluxes=True it returns ln L (dropped here, the name _) and the
# fluxes
_, flux_at_truth = joint.loglike_profiled(truth, windows, psi, inst, sigma_noise, return_fluxes=True)
print(f"profiled fluxes at the truth   {flux_at_truth.round(1)}   (injected {scene.fluxes})")
for label, theta in (("truth", truth), ("seed (sinogram)", seed)):
    lnl = joint.loglike_profiled(theta, windows, psi, inst, sigma_noise)
    print(f"ln L profiled at the {label:16s} {lnl:.3f} nats")

In [ ]:
# the three forms are one quantity: the full, the profiled, and chi^2 (the constant is N/2 ln 2 pi sigma_n^2)
# N_tot: the number of samples of all the windows together
N_tot = 0
for window in windows:
    N_tot = N_tot + window.size
# form 1 against form 2, at the truth: ln L at the profiled fluxes given by hand is the profiled ln L
lnl_full_truth = joint.loglike_windows(truth, flux_at_truth, windows, psi, inst, sigma_noise)
lnl_profiled_truth = joint.loglike_profiled(truth, windows, psi, inst, sigma_noise)
assert np.isclose(lnl_full_truth, lnl_profiled_truth, rtol=1e-12)
# form 3, at the seed: the residual of all the samples from the stacked model A f. np.ravel(windows) is the
# samples of all the windows in one long array; joint.stacked_design is A(theta), one row per sample; @ is the
# matrix product
lnl_seed, flux_at_seed = joint.loglike_profiled(seed, windows, psi, inst, sigma_noise, return_fluxes=True)
residual_seed = np.ravel(windows) - joint.stacked_design(seed, psi, inst) @ flux_at_seed
# residual_seed @ residual_seed is the summed squared residual
assert np.isclose(lnl_seed, -0.5 * (residual_seed @ residual_seed) / sigma_noise**2
                  - 0.5 * N_tot * np.log(2 * np.pi * sigma_noise**2), rtol=1e-12)

## 7. The fit (Sect. 4.3)

The maximum of the profiled likelihood over θ, started at the seed of step 5 (a bounded
Nelder–Mead search, restarted from its answer until it no longer improves). At the solution: the
fluxes, the Fisher matrix of the two offsets and two fluxes, and its inverse, the covariance.

In [ ]:
# the fit: maximize the profiled ln L over theta, from the seed
# the bounds: each of the two offsets within +/- joint.THETA_BOUND [mas] (a list times 2 repeats its entry)
bounds = [(-joint.THETA_BOUND, joint.THETA_BOUND)] * 2


# the optimizer minimises, so it is given -ln L: its minimum is the maximum of the profiled ln L
def negative_loglike(theta_trial):
    return -joint.loglike_profiled(theta_trial, windows, psi, inst, sigma_noise)


# opt.minimize_restarts: bounded Nelder-Mead, restarted from its answer until a restart stops improving it. It
# returns the result (result.x is the solution), the number of restarts, and whether it converged
result, n_restarts, converged = opt.minimize_restarts(negative_loglike, seed, bounds=bounds,
                                                      xatol=1e-4, maxiter_first=4000, maxiter_restart=16000)

In [ ]:
# at the solution: the fluxes, chi^2, the Fisher matrix and the covariance
theta_hat = np.asarray(result.x)                                    # the fitted offset (dra*, ddec) [mas]
lnl_hat, flux_hat = joint.loglike_profiled(theta_hat, windows, psi, inst, sigma_noise, return_fluxes=True)
# rnorm^2 / sigma_n^2, Eq. 18 without its constant
chi2_hat = -2.0 * lnl_hat - N_tot * np.log(2 * np.pi * sigma_noise**2)
# joint.joint_fisher, 4 x 4: (dalpha*, ddelta, f_primary, f_companion)
fisher = joint.joint_fisher(theta_hat, flux_hat, psi, inst, sigma_noise)
# joint.joint_covariance: its inverse, and the 1-sigma errors of theta (the first theta_hat.size entries)
cov, sigma_theta = joint.joint_covariance(fisher, theta_hat.size)

assert converged
print(f"restarts {n_restarts}, converged {converged}")
print(f"ln L at the maximum            {lnl_hat:.3f} nats")
print(f"chi2 / N samples               {chi2_hat:.1f} / {N_tot}")

In [ ]:
# the convenience: joint.fit_joint makes the same fit in one call
# it returns a dictionary with the entries "theta", "flux", "cov" and "sigma_theta" (among others)
fit_one_call = joint.fit_joint(windows, psi, seed, inst, sigma_noise)
assert np.array_equal(fit_one_call["theta"], theta_hat) and np.array_equal(fit_one_call["flux"], flux_hat)
assert np.array_equal(fit_one_call["cov"], cov) and np.array_equal(fit_one_call["sigma_theta"], sigma_theta)
print("joint.fit_joint reproduces the solution and its covariance exactly")

## 8. The uncertainties, and the inset of panel (c) (Sect. 4.4)

The offsets and 𝒮 with their 1σ errors, from the covariance. The truth's distance from the
solution is read on the likelihood: $2\Delta\ln\mathcal L$ is a Δχ² with two degrees of freedom,
whose 1, 2, 3σ levels (2.30, 6.18, 11.83), the joint 68.3, 95.4 and 99.7 % regions of the two
offsets, the inset draws. One draw cannot test the errors;
the coverage over many draws is `auxil/sim_coverage_fig3.ipynb`.

In [ ]:
# the errors of the offsets and of S, the truth's Delta chi^2, and the numbers of Sect. 4.4 asserted
# step 1, the flux ratio S = f_companion / f_primary
flux_primary, flux_companion = flux_hat
q_hat = flux_companion / flux_primary
# step 2, its error by linear propagation. cov is 4 x 4 in the order (dalpha*, ddelta, f_primary, f_companion);
# [2:, 2:] keeps the last two rows and columns, the fluxes' block: [0, 0] the primary's variance, [1, 1] the
# companion's, [0, 1] their covariance
cov_flux = cov[2:, 2:]
sig_q = q_hat * np.sqrt(cov_flux[1, 1] / flux_companion**2 + cov_flux[0, 0] / flux_primary**2
                        - 2 * cov_flux[0, 1] / (flux_primary * flux_companion))
# step 3, the truth's distance from the solution: twice the difference of the profiled ln L
lnl_at_solution = joint.loglike_profiled(theta_hat, windows, psi, inst, sigma_noise)
lnl_at_truth = joint.loglike_profiled(truth, windows, psi, inst, sigma_noise)
dchi2_truth = 2 * (lnl_at_solution - lnl_at_truth)

print(f"offset  ({theta_hat[0]:.1f} +/- {sigma_theta[0]:.1f}, {theta_hat[1]:.1f} +/- {sigma_theta[1]:.1f}) mas;  "
      f"truth ({companion_ra:.1f}, {companion_dec:.1f})")
print(f"q       {q_hat:.4f} +/- {sig_q:.4f}   (truth {Q})")
print(f"truth at Delta chi2 = {dchi2_truth:.2f}  (1 sigma for two parameters: 2.30)")
assert np.array_equal(np.round(theta_hat, 1), [189.8, -69.0])            # Sect. 4.4
# np.hypot(*(theta_hat - truth)) is the distance of the solution from the truth on the sky [mas]
assert round(float(np.hypot(*(theta_hat - truth)))) == 2                  # "within approximately 2 mas"
assert np.array_equal(np.round(sigma_theta, 1), [1.3, 0.9])
assert round(q_hat, 3) == 0.197 and round(sig_q, 3) == 0.003
assert round(dchi2_truth, 1) == 3.3

In [ ]:
# the inset: the profiled Delta chi^2 on a grid around the solution
INSET_HALF = 4.0                           # half the side of the inset's grid [mas]
INSET_NGRID = 31                           # the number of grid points on each side
grid_ra = np.linspace(theta_hat[0] - INSET_HALF, theta_hat[0] + INSET_HALF, INSET_NGRID)    # the grid's dra* [mas]
grid_dec = np.linspace(theta_hat[1] - INSET_HALF, theta_hat[1] + INSET_HALF, INSET_NGRID)   # the grid's ddec [mas]
# the profiled ln L at every grid point: one row per ddec, one column per dra*
lnl_grid = []
for dec in grid_dec:
    lnl_row = []
    for ra in grid_ra:
        lnl_row.append(joint.loglike_profiled([ra, dec], windows, psi, inst, sigma_noise))
    lnl_grid.append(lnl_row)
dchi2 = -2.0 * np.array(lnl_grid)                                         # rows: dec, columns: ra*
dchi2 -= dchi2.min()                                                      # Delta chi^2 from the grid's best point

## 9. The two transits the figure highlights

The best-measured resolved transit and a typical blended one (projected separation above and
below one FWHM). The choice uses the truth and is for presentation only.

In [ ]:
# the two transits to highlight: the best-measured resolved one and a typical blended one (chosen with the truth,
# for presentation only)
true_proj = ins.project_al(companion_ra, companion_dec, det_psi)    # the companion's true eta at each detection [mas]
# a detection is the companion's when it is within 3 sigma of the true projection, or within 5 mas when 3 sigma
# is less than 5 mas
is_companion = np.abs(det_eta - true_proj) < np.maximum(3 * det_sigma, 5.0)
# np.flatnonzero gives the indices of the detections for which the test holds
blended = np.flatnonzero(is_companion & (np.abs(true_proj) <= FWHM_MAS))    # projected separation up to one FWHM
resolved = np.flatnonzero(is_companion & (np.abs(true_proj) > FWHM_MAS))    # projected separation above one FWHM
median_sigma_blended = np.quantile(det_sigma[blended], 0.5)              # the median sigma_eta of the blended ones
# the resolved detection with the smallest sigma_eta, and the blended one whose sigma_eta is nearest the median
best_resolved = int(resolved[np.argmin(det_sigma[resolved])])
typical_blended = int(blended[np.argmin(np.abs(det_sigma[blended] - median_sigma_blended))])
highlighted = [best_resolved, typical_blended]                          # the two detections' indices

for name, i in zip(("resolved", "blended"), highlighted):               # i: the detection's index
    print(f"{name:9s} window {det_window[i]:2d}, psi {np.rad2deg(det_psi[i]) % 360:5.1f} deg, "
          f"eta {det_eta[i]:6.1f} +/- {det_sigma[i]:4.1f} mas")

## 10. The figure (Fig. 2)

(a) The scan directions, two highlighted, with the drop from the companion onto each scan axis
and the measured η̂. (b) The sinogram, the injected track ρ cos(ψ − φ) dashed, and the gray band
where the noise-free blend has one peak (for 𝒮 = 0.2 about 3.1 σ_LSF, solved below). (c) The
triangulation map with the fit's Δχ² contours in the inset. Plotting only.

In [ ]:
# the figure: (a) the scan directions, (b) the sinogram, (c) the triangulation map with the fit's inset
def bimodal_limit(q, sigma_lsf=FWHM_MAS / (2.0 * np.sqrt(2.0 * np.log(2.0)))):
    """Separation [mas] below which a pair of Gaussians of flux ratio q has one peak.

    Inputs: q, the flux ratio; sigma_lsf, the Gaussians' sigma [mas] (the default is the LSF's)."""
    def two_peaks(separation):                                            # count the local maxima of the noise-free blend
        x = np.linspace(-4 * sigma_lsf - separation, 4 * sigma_lsf + separation, 8001)   # a fine grid of eta [mas]
        y = np.exp(-0.5 * (x / sigma_lsf) ** 2) + q * np.exp(-0.5 * ((x - separation) / sigma_lsf) ** 2)   # the blend
        # y[1:-1] is every point but the two ends; a local maximum is higher than the point before it (y[:-2])
        # and the point after it (y[2:])
        return ((y[1:-1] > y[:-2]) & (y[1:-1] > y[2:])).sum() >= 2
    lo = 0.1                                                              # a separation with one peak [mas]
    up = 20.0 * sigma_lsf                                                 # a separation with two peaks [mas]
    for _ in range(50):                                                   # bisection on the separation, 50 halvings
        mid = 0.5 * (lo + up)
        if two_peaks(mid):
            up = mid
        else:
            lo = mid
    return up


plt.rcParams.update({"font.size": 10.0, "axes.linewidth": 0.5,
                     "xtick.labelsize": 9.0, "ytick.labelsize": 9.0})
ETA_PANEL_MAS = 280.0                      # panel (b) eta range
BAR_CAP_MAS = 80.0                         # drawn error-bar half-length cap
BLEND_LIMIT_MAS = bimodal_limit(Q)         # below this separation the noise-free pair has one peak [mas]
pa_rad = np.deg2rad(PA_DEG)                # the companion's position angle [rad]
highlight_colors = ["tomato", "dodgerblue"]   # the resolved transit, the blended transit
# COMP: the marker of the true companion, a table of plot options passed to ax.plot as **COMP
COMP = dict(marker="*", color="white", ms=12, mec="k", mew=0.6, zorder=6)
SKY_HALF_RANGE_MAS = 350.0                 # sky half-range of panels (a) and (c) [mas]
LINE_HALF_LENGTH_MAS = 800.0               # drawn line half-length, cropped by the axes

fig, (ax_g, ax_s, ax_m) = plt.subplots(1, 3, figsize=(10.6, 3.6))   # the panels (a), (b), (c)

# (a) every scan direction through the source, two transits highlighted
for psi_j in psi:
    u_ra = np.sin(psi_j)                   # (u_ra, u_dec): the unit vector along the scan
    u_dec = np.cos(psi_j)
    ax_g.plot([-LINE_HALF_LENGTH_MAS * u_ra, LINE_HALF_LENGTH_MAS * u_ra],
              [-LINE_HALF_LENGTH_MAS * u_dec, LINE_HALF_LENGTH_MAS * u_dec],
              color="0.8", lw=0.45, alpha=0.35, zorder=1)
for k, color in zip(highlighted, highlight_colors):                 # k: the index of a highlighted detection
    psi_j = det_psi[k]
    eta_measured = det_eta[k]
    u_ra = np.sin(psi_j)
    u_dec = np.cos(psi_j)
    ax_g.plot([-LINE_HALF_LENGTH_MAS * u_ra, LINE_HALF_LENGTH_MAS * u_ra],
              [-LINE_HALF_LENGTH_MAS * u_dec, LINE_HALF_LENGTH_MAS * u_dec],
              color=color, lw=8, alpha=0.22, solid_capstyle="butt", zorder=2)
    # dotted: the perpendicular drop to the true projection; solid: from there
    # along the scan to the measured eta-hat (the measurement error)
    eta_foot = companion_ra * u_ra + companion_dec * u_dec          # Eq. 9: the companion's true eta on this scan
    ax_g.plot([companion_ra, eta_foot * u_ra], [companion_dec, eta_foot * u_dec],
              ls=":", lw=0.9, color=color, zorder=3)
    ax_g.plot([eta_foot * u_ra, eta_measured * u_ra], [eta_foot * u_dec, eta_measured * u_dec],
              ls="-", lw=0.9, color=color, zorder=3)
    ax_g.plot(eta_measured * u_ra, eta_measured * u_dec, "o", ms=4, color=color, zorder=4)
    ax_g.annotate(rf"$\psi={np.rad2deg(psi_j) % 360:.0f}^\circ$",
                  (0.86 * SKY_HALF_RANGE_MAS * u_ra, 0.86 * SKY_HALF_RANGE_MAS * u_dec), fontsize=8,
                  color=color, ha="center", va="center", zorder=5)
ax_g.plot(0, 0, "+", color="k", ms=10, mew=1.2, zorder=5)
ax_g.plot(companion_ra, companion_dec, **COMP)
ax_g.set_xlim(SKY_HALF_RANGE_MAS, -SKY_HALF_RANGE_MAS)
ax_g.set_ylim(-SKY_HALF_RANGE_MAS, SKY_HALF_RANGE_MAS)
ax_g.set_xlabel(r"$\Delta\alpha^{*}$  [mas]   (East $\leftarrow$)")
ax_g.set_ylabel(r"$\Delta\delta$  [mas]")
ax_g.tick_params(direction="in")

# (b) the sinogram, the blend band, the two highlighted transits
psi_fine = np.linspace(0, 360, 361)        # the scan angles of the drawn track [deg]
ax_s.axhspan(-BLEND_LIMIT_MAS, BLEND_LIMIT_MAS, color="0.935", lw=0, zorder=0)
ax_s.plot(psi_fine, RHO_MAS * np.cos(np.deg2rad(psi_fine) - pa_rad), ls="--",
          lw=0.9, color="0.55", zorder=1)
det_psi_deg = np.rad2deg(det_psi) % 360.0  # the detections' scan angles [deg, 0 to 360]
ax_s.errorbar(det_psi_deg, det_eta, yerr=np.minimum(det_sigma, BAR_CAP_MAS), fmt="o", ms=3.5,
              color="k", elinewidth=0.3, capsize=0.0, zorder=2)
capped = det_sigma > BAR_CAP_MAS           # the detections whose error bar is drawn capped
if capped.any():
    # an arrow head at each end of a capped bar: up at +BAR_CAP_MAS, down at -BAR_CAP_MAS
    for sign, mark in ((+1.0, "^"), (-1.0, "v")):
        ax_s.plot(det_psi_deg[capped], det_eta[capped] + sign * BAR_CAP_MAS, mark, ms=3.2,
                  color="k", ls="none", zorder=2.6)
print(f"bars off panel: {int(((det_eta + det_sigma) > ETA_PANEL_MAS).sum())} up, "
      f"{int(((det_eta - det_sigma) < -ETA_PANEL_MAS).sum())} down; largest sigma_eta "
      f"{det_sigma.max():.0f} mas -> weight {(det_sigma.min() / det_sigma.max()) ** 2:.1e} "
      f"of the best detection's")
for k, color in zip(highlighted, highlight_colors):
    ax_s.errorbar(np.rad2deg(det_psi[k]) % 360.0, det_eta[k], yerr=det_sigma[k],
                  fmt="o", ms=9, color=color, mec="k", mew=0.4,
                  elinewidth=1, capsize=1.5, zorder=3)
ax_s.set_xlim(0, 360)
ax_s.set_xticks([0, 90, 180, 270, 360])
ax_s.set_xlabel(r"scan angle $\psi$  [deg]")
ax_s.set_ylabel(r"detected $\hat\eta$  [mas]")
ax_s.set_ylim(-ETA_PANEL_MAS, ETA_PANEL_MAS)
ax_s.tick_params(direction="in")

# (c) the back-projection map, saturated so the stacking shows; the fit's inset
sky_map, map_axis, extent = bp.backprojection_map(det_psi, det_eta, det_sigma, MAP_HALF_SIZE_MAS, MAP_STEP_MAS)
ax_m.imshow(sky_map, origin="lower", extent=extent, aspect="auto", cmap="gray_r",
            vmin=0.0, vmax=0.70 * sky_map.max())   # saturated at 70 % of the peak, so the single ridges show
ax_m.plot(0, 0, "+", color="k", ms=10, mew=1.2, zorder=5)
ax_m.plot(companion_ra, companion_dec, **COMP)
ax_m.set_xlim(map_axis[-1], map_axis[0])   # East to the left
ax_m.set_xlabel(r"$\Delta\alpha^{*}$  [mas]   (East $\leftarrow$)")
ax_m.set_ylabel(r"$\Delta\delta$  [mas]")
ax_m.tick_params(direction="in", color="0.5")

# the inset: the contours of the fit's Delta chi^2 at 1, 2, 3 sigma for two parameters; the solution (cross),
# the seed (open circle) and the truth (star)
ax_inset = ax_m.inset_axes([0.58, 0.58, 0.40, 0.40])
contours = ax_inset.contour(grid_ra, grid_dec, dchi2, levels=[2.30, 6.18, 11.83],
                            colors="k", linewidths=[1.0, 0.7, 0.5])
ax_inset.clabel(contours, fmt={2.30: r"1$\sigma$", 6.18: r"2$\sigma$",
                               11.83: r"3$\sigma$"}, fontsize=6, inline=True)
ax_inset.plot(*theta_hat, "x", color="k", ms=5, mew=1.1, zorder=5)
ax_inset.plot(seed[0], seed[1], "o", mfc="none", mec="tomato", ms=5, mew=0.9, zorder=4)
ax_inset.plot(companion_ra, companion_dec, **{**COMP, "ms": 8})   # COMP with its marker size set to 8
ax_inset.set_xlim(theta_hat[0] + INSET_HALF, theta_hat[0] - INSET_HALF)   # East to the left
ax_inset.set_ylim(theta_hat[1] - INSET_HALF, theta_hat[1] + INSET_HALF)
ax_inset.locator_params(nbins=3)
ax_inset.tick_params(direction="in", length=2, labelsize=5.5, pad=1.5)
for spine in ax_inset.spines.values():
    spine.set_edgecolor("k")
    spine.set_linewidth(0.8)

for ax, letter in zip((ax_g, ax_s, ax_m), "abc"):
    ax.set_box_aspect(1)
    ax.text(0.03, 0.97, f"({letter})", transform=ax.transAxes, va="top",
            ha="left", fontsize=9, fontweight="bold")
fig.tight_layout(pad=0.5, w_pad=1.0)

In [ ]:
# write the PDF only when SAVE_PDF is True
if SAVE_PDF:
    Path("pdf").mkdir(exist_ok=True)
    fig.savefig(Path("pdf") / "fig2_scans_to_sky.pdf", bbox_inches="tight")
    print("wrote pdf/fig2_scans_to_sky.pdf")
else:
    print("SAVE_PDF is False: the PDF is not written")